# Exam Preparation RAG

A clean, reproducible notebook for the PDF-based Exam Preparation RAG system.

The implementation lives in `rag_engine.py`; this notebook is a demonstration layer, not a second copy of the application logic.

## 1. Install dependencies

Run this once in the project environment. Restart the kernel only if packages were newly installed.

In [ ]:
%pip install -r requirements-rag.txt

## 2. Imports and configuration

The project uses `pymupdf` for PDF extraction, deterministic TF-IDF/SVD retrieval, and Gemini through the OpenAI-compatible API.

In [ ]:
from pathlib import Path
import os

from rag_engine import (
    DOCUMENTS_DIR,
    build_context,
    clear_knowledge_base_cache,
    create_chunks,
    create_index,
    load_pdfs,
    retrieve,
)


## 3. Inspect the study PDFs

In [ ]:
pdf_files = sorted(DOCUMENTS_DIR.glob('*.pdf'))
if not pdf_files:
    raise FileNotFoundError(f'No PDF files found in {DOCUMENTS_DIR}')

print(f'Found {len(pdf_files)} PDF file(s):')
for path in pdf_files:
    print(f'- {path.name}')


## 4. Load and chunk the documents

In [ ]:
documents = load_pdfs(DOCUMENTS_DIR)
chunks = create_chunks(documents, chunk_size=800, overlap=100)

print(f'Loaded {len(documents)} non-empty PDF pages')
print(f'Created {len(chunks)} retrieval chunks')


## 5. Build the retrieval index

In [ ]:
vectorizer, svd, normalizer, index = create_index(chunks)
print(f'Index shape: {index.shape}')
print(f'SVD enabled: {svd is not None}')


## 6. Test retrieval before calling the LLM

In [ ]:
question = 'What is normalization?'
results = retrieve(
    question,
    chunks,
    vectorizer,
    svd,
    normalizer,
    index,
    k=3,
)

for i, result in enumerate(results, start=1):
    print('=' * 70)
    print(f'{i}. {result["source"]} | page {result["page"]} | score {result["score"]:.4f}')
    print(result['text'][:500])


## 7. Inspect the exact context sent to the generator

In [ ]:
context = build_context(results)
print(context)


## 8. Generate a grounded answer

Set `GOOGLE_API_KEY` in the repository `.env` file before running this section. The model is instructed to answer only from retrieved study material.

In [ ]:
from rag_engine import exam_rag

if not os.getenv('GOOGLE_API_KEY'):
    print('GOOGLE_API_KEY is not configured; retrieval-only sections above are still fully testable.')
else:
    result = exam_rag('What is normalization?', mode='explain', k=3)
    print(result['answer'])
    print('\nSources:')
    for source in result['sources']:
        print(f'- {source["source"]} | page {source["page"]} | score {source["score"]:.4f}')


## 9. Exam modes

- `explain`: clear student explanation
- `short`: concise exam answer
- `5-mark`: structured 5-mark answer
- `mcq`: five multiple-choice questions


In [ ]:
if os.getenv('GOOGLE_API_KEY'):
    for mode in ('short', '5-mark', 'mcq'):
        result = exam_rag('Explain normalization.', mode=mode, k=3)
        print(f'\n--- {mode} ---\n{result["answer"]}')
else:
    print('Configure GOOGLE_API_KEY to run generation modes.')


## 10. Rebuild after changing PDFs

The application caches the local knowledge base for performance. Clear that cache after adding or replacing study PDFs.

In [ ]:
clear_knowledge_base_cache()
print('Knowledge-base cache cleared.')


## API

From the repository root:

```bash
uvicorn exam_preparation_rag.app:app --reload
```

Then open `/docs` for Swagger UI.